## 05-supervisor.ipynb

### 멀티 에이전트
멀티 에이전트 패턴 중 하나
- 도메인이 여러개 섞여 있고
- 각 도메인마다 도구가 많고 복잡함
- 하위 담당 에이전트와 사용자가 소통할 필요가 없음
- 단순 도구만 활용할 경우에는 사용 X

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
# from langchain_openai import ChatOpenAI
from langchain.chat_models import init_chat_model # 위 기능과 비슷하지만 가이드 용으로 시작.

model = init_chat_model('gpt-4.1-mini')

In [6]:
from langchain.tools import tool

@tool
def create_calendar_event(
    title: str,
    start_time: str,
    end_time: str,
    attendees: list[str], # ['a@a.com', 'b@b.com']
    location: str = '', # 위치가 없을 경우, 빈 문자열
):
    '''캘린더 이벤트 생성'''

    return f'이벤트 생성 완료. {title} - {start_time} ~ {end_time}.'
    

@tool
def send_email(
    to: list[str],
    subject: str,
    body: str
):
    '''이메일 발송'''
    return f'이메일 발송 완료. {to} - {subject}'
    
@tool
def get_available_time_slot(
        attendees: list[str],
    date: str,
    duration_minutes: int
):
    '''참가자들이 특정 날짜에 참여 가능한 시간 확인'''
    return ['09:00', '14:00', '16:00']


In [13]:
from langchain.agents import create_agent
from datetime import datetime
CALENDAR_AGENT_PROMPT = (
"You are a calendar scheduling assistant. "
    "Parse natural language scheduling requests (e.g., 'next Tuesday at 2pm') "
    "into proper ISO datetime formats. "
    "Use get_available_time_slots to check availability when needed. "
    "Use create_calendar_event to schedule events. "
    "Always confirm what was scheduled in your final response."
    f"NOW: {datetime.now()}"
)
calendar_agent = create_agent(
    model,
    tools=[create_calendar_event, get_available_time_slot],
    system_prompt=CALENDAR_AGENT_PROMPT
)

In [ ]:
query = '다음주 화요일 2시에 1시간 동안 팀 미팅을 잡아줘'

for step in calendar_agent.stream(
    {'messages': [{'role': 'user', 'content': query}]}
):
    for update in step.values():
        for message in update.get('messages', []):
            message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  get_available_time_slot (call_H14lcamwqVjUBC2m52DWQL9n)
 Call ID: call_H14lcamwqVjUBC2m52DWQL9n
  Args:
    attendees: ['team']
    date: 2026-03-03
    duration_minutes: 60
================================= Tool Message =================================
Name: get_available_time_slot

["09:00", "14:00", "16:00"]
================================== Ai Message ==================================
Tool Calls:
  create_calendar_event (call_EXwRExxsUE5l3BWo2ddzSWEo)
 Call ID: call_EXwRExxsUE5l3BWo2ddzSWEo
  Args:
    title: 팀 미팅
    start_time: 2026-03-03T14:00:00
    end_time: 2026-03-03T15:00:00
    attendees: ['team']
    location:
================================= Tool Message =================================
Name: create_calendar_event

이벤트 생성 완료. 팀 미팅 - 2026-03-03T14:00:00 ~ 2026-03-03T15:00:00.
================================== Ai Message ==================================

다음주 화요일(3월 3일) 오후

In [15]:
EMAIL_AGENT_PROMPT = (
    "You are an email assistant. "
    "Compose professional emails based on natural language requests. "
    "Extract recipient information and craft appropriate subject lines and body text. "
    "Use send_email to send the message. "
    "Always confirm what was sent in your final response."
)

email_agent = create_agent(
    model,
    tools=[send_email],
    system_prompt=EMAIL_AGENT_PROMPT,
)

In [ ]:
query = '디자인 팀한테 내일 오전 10시에 있는 디자인 리뷰 리마인더 보내줘'

for step in email_agent.stream(
    {'messages': [{'role': 'user', 'content': query}]}
):
    for update in step.values():
        for message in update.get('messages', []):
            message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  send_email (call_pUohczcJwRZ68NgiXMdSJyth)
 Call ID: call_pUohczcJwRZ68NgiXMdSJyth
  Args:
    to: ['디자인팀']
    subject: 다음주 화요일 디자인팀 미팅 일정 안내
    body: 안녕하세요 디자인팀 여러분,

다음주 화요일 오전 10시부터 1시간 동안 디자인팀 미팅이 예정되어 있습니다. 일정 참고하시어 참석 부탁드립니다.

감사합니다.
================================= Tool Message =================================
Name: send_email

이메일 발송 완료. ['디자인팀'] - 다음주 화요일 디자인팀 미팅 일정 안내
================================== Ai Message ==================================

다음주 화요일 오전 10시부터 1시간 동안 디자인팀 미팅 일정을 디자인팀에 안내하는 이메일을 보냈습니다. 추가 요청이 있으시면 알려주세요.


In [ ]:
# SQL 데이터를 langchain에 연결해보기-1
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit

import os

DB_URI = os.environ.get('DB_URI')

db = SQLDatabase.from_uri(DB_URI)
toolkit = SQLDatabaseToolkit(db=db, llm=model)

dialect = db.dialect
top_k = 5

SQL_AGENT_PROMPT = f"""
You are an agent designed to interact with a SQL database.
Given an input question, create a syntactically correct {dialect} query to run,
then look at the results of the query and return the answer. Unless the user
specifies a specific number of examples they wish to obtain, always limit your
query to at most {top_k} results.

You can order the results by a relevant column to return the most interesting
examples in the database. Never query for all the columns from a specific table,
only ask for the relevant columns given the question.

You MUST double check your query before executing it. If you get an error while
executing a query, rewrite the query and try again.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the
database.

To start you should ALWAYS look at the tables in the database to see what you
can query. Do NOT skip this step.

Then you should query the schema of the most relevant tables.
"""

db_agent = create_agent(
    model, 
    toolkit.get_tools(), 
    system_prompt=SQL_AGENT_PROMPT,
)

In [ ]:
# SQL 데이터를 langchain에 연결해보기-2
query = '백엔드 팀에 있는 사람들의 이메일주소만 알려줘'

for step in db_agent.stream(
    {'messages': [{'role': 'user', 'content': query}]}
):
    for update in step.values():
        for message in update.get('messages', []):
            message.pretty_print()

In [ ]:
@tool
def schedule_event(request: str) -> str:
    """Schedule calendar events using natural language.

    Use this when the user wants to create, modify, or check calendar appointments.
    Handles date/time parsing, availability checking, and event creation.

    Input: Natural language scheduling request (e.g., 'meeting with design team
    next Tuesday at 2pm')
    """
    result = calendar_agent.invoke({
        "messages": [{"role": "user", "content": request}]
    })
    return result["messages"][-1].text


@tool
def manage_email(request: str) -> str:
    """Send emails using natural language.

    Use this when the user wants to send notifications, reminders, or any email
    communication. Handles recipient extraction, subject generation, and email
    composition.

    Input: Natural language email request (e.g., 'send them a reminder about
    the meeting')
    """
    result = email_agent.invoke({
        "messages": [{"role": "user", "content": request}]
    })
    return result["messages"][-1].text


@tool
def query_db(request: str) -> str:
    """Query Database using natural language.

    Team info and employee info are in Database.
    Use this when you need to query DB.

    Input: Natural language query request (e.g., 'members in data team.')
    """
    result = db_agent.invoke({
        "messages": [{"role": "user", "content": request}]
    })
    return result["messages"][-1].text

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

SUPERVISOR_PROMPT = '''너는 매우 유능한 개인 비서야.
너는 아래와 같은 일을 할 수있어.

1. DB에서 팀, 멤버 정보 쿼리.
2. 캘린더 이벤트를 조정
3. 이메일을 전송 (DB에서 메일 주소 참조 필요)

사용자의 요청을 분석하여, 적절한 도구를 사용하고, 결과를 종합해야해.
요청이 여러가지 액션을 취해야 하면, 순서를 잘 짜서 각종 도구들을 여러번 호출해.
'''
supervisor_model = init_chat_model('gpt-4.1-mini')

supervisor_agent = create_agent(
    supervisor_model,
    tools=[schedule_event, manage_email, query_db],
    system_prompt=SUPERVISOR_PROMPT,
    checkpointer=InMemorySaver()
)

In [ ]:

query = '아까 한거 오후 4시로 바꾸자'

config = {'configurable': {'thread_id': '12345d6'}}


for step in supervisor_agent.stream(
    {'messages': [{'role': 'user', 'content': query}]},
    config=config,
):
    for update in step.values():
        for message in update.get('messages', []):
            message.pretty_print()

In [ ]:
# SQL 적용 실습 전에 적용했던 코드들
@tool
def schedule_event(request: str) -> str:
    """Schedule calendar events using natural language.

    Use this when the user wants to create, modify, or check calendar appointments.
    Handles date/time parsing, availability checking, and event creation.

    Input: Natural language scheduling request (e.g., 'meeting with design team
    next Tuesday at 2pm')
    """
    result = calendar_agent.invoke({
        "messages": [{"role": "user", "content": request}]
    })
    return result["messages"][-1].text


@tool
def manage_email(request: str) -> str:
    """Send emails using natural language.

    Use this when the user wants to send notifications, reminders, or any email
    communication. Handles recipient extraction, subject generation, and email
    composition.

    Input: Natural language email request (e.g., 'send them a reminder about
    the meeting')
    """
    result = email_agent.invoke({
        "messages": [{"role": "user", "content": request}]
    })
    return result["messages"][-1].text # 마지막 메세지만 관리자 에이전트에게 전달

In [22]:
SUPERVISOR_PROMPT= '''너는 매우 유능한 개인 비서야.
너는 캘린더 이벤트를 조정하고, 이메일을 보낼 수 있어.
사용자의 요청을 분석하여, 적절한 도구를 사용하고, 결과를 종합해야해.
요청이 여러가지 액션을 취해야 하면, 순서를 잘 짜서 각종 도구들을 여러번 호출해.
'''

supervisor_agent = create_agent(
    model,
    tools=[schedule_event, manage_email],
    system_prompt=SUPERVISOR_PROMPT,
)

In [24]:
query = '디자인 팀에는 alice, bob, charile 가 있어. 다음주 화요일 오후 2시부터 1시간동안 디자인 팀 미팅 잡고 메일 보내놔'

for step in supervisor_agent.stream(
    {'messages': [{'role': 'user', 'content': query}]}
):
    for update in step.values():
        for message in update.get('messages', []):
            message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  schedule_event (call_wuChHojCZSVztbV1VTwZxy7o)
 Call ID: call_wuChHojCZSVztbV1VTwZxy7o
  Args:
    request: design team meeting next Tuesday at 2pm for 1 hour
  manage_email (call_6X5jpJdMCrewwEqVZgLAEjpg)
 Call ID: call_6X5jpJdMCrewwEqVZgLAEjpg
  Args:
    request: send an email to alice, bob, and charile notifying about the design team meeting scheduled next Tuesday at 2pm for 1 hour
================================= Tool Message =================================
Name: manage_email

I have sent an email to Alice, Bob, and Charlie notifying them about the design team meeting scheduled for next Tuesday at 2 PM for 1 hour. If you need any further assistance, please let me know.
================================= Tool Message =================================
Name: schedule_event

The design team meeting has been scheduled for next Tuesday, March 3rd, from 2:00 PM to 3:00 PM.
===================